In [45]:
import andes
import numpy as np
import pandas as pd
import pandapower as pp

import matplotlib
matplotlib.use('TkAgg')
import matplotlib.pyplot as plt

MODEL_pp = 'pp_PFlow_2035'
MODEL_andes = 'andes_PFlow_2035'


### Andes powerflow

In [46]:
ss = andes.main.load(MODEL_andes+'.xlsx', setup=False, config_path = "config_file.rc")
ss.setup()
ss.PFlow.run()

Sb = ss.config.mva

# READ RESULTS .txt FILE
with open(MODEL_andes+'_out.txt', 'r') as file:
    lines = file.readlines()

line_data_start = lines.index('LINE DATA:\n') + 3
line_data_end = lines.index('OTHER ALGEBRAIC VARIABLES:\n') -1
line_data = lines[line_data_start:line_data_end]

data = []
for line in line_data:
    parts = line.split()
    idx_first_numeric = next(i for i, x in enumerate(parts) if x.replace('.','',1).isdigit())
    name = " ".join(parts[:idx_first_numeric])
    row = [name] + [float(x) for x in parts[idx_first_numeric:]]
    data.append(row)

columns = ["Line Name", "From Bus", "To Bus", "P From", "Q From", "P To", "Q To"]
df = pd.DataFrame(data, columns=columns)

### Pandapower powerflow

In [47]:
net = pp.from_excel(MODEL_pp+'.xlsx')
pp.runpp(net)

numba cannot be imported and numba functions are disabled.
Probably the execution is slow.
Please install numba to gain a massive speedup.
(or if you prefer slow execution, set the flag numba=False to avoid this warning!)



### Bus

In [48]:
bus_V_nom = np.asarray(ss.Bus.as_df(vin=True)['Vn'].tolist())

pp_V_pu = net.res_bus.vm_pu.to_list()
andes_V_pu = (np.array(ss.Bus.v.v)*380)/ bus_V_nom

pp_V_kV = pp_V_pu * bus_V_nom
andes_V_kV = andes_V_pu * bus_V_nom

print(pp_V_pu[12])
print(andes_V_pu[12])

x = np.arange(len(pp_V_pu))
w = 0.35
plt.figure(figsize=(max(8, 0.35*len(x)), 4))
plt.grid()
plt.bar(x - w/2, pp_V_pu, width=w, label='pandapower')
plt.bar(x + w/2, andes_V_pu, width=w, label='ANDES')
plt.xticks(x, ss.Bus.name.v, rotation=90, fontsize=8)
plt.xlabel("Bus Name")
plt.ylabel("Voltage (kV)")
plt.title('Plot bus voltages')
plt.legend()
#plt.savefig('afb/V_mag_bus.svg')
plt.show()

1.0
1.0000000000000002


In [49]:
pp_Va = ((2*np.pi)/360)*np.array(net.res_bus.va_degree.to_list())
andes_Va = np.array(ss.Bus.a.v)

print(pp_Va[1])
print(andes_Va[1])

x = np.arange(len(pp_Va))
w = 0.35

plt.figure(figsize=(max(8, 0.35*len(x)), 4))
plt.grid()
plt.bar(x - w/2, pp_Va, width=w, label='pandapower')
plt.bar(x + w/2, andes_Va, width=w, label='ANDES')
plt.xlabel("Bus Name")
plt.ylabel("Voltage angle (rad)")
plt.title('Plot bus voltage angles')
plt.legend()
#plt.savefig('afb/V_angle_bus.svg')
plt.show()

0.2353952395145558
0.24285069464099823

### Generator

In [50]:
pp_p = net.res_gen.p_mw.to_list() + net.res_sgen.p_mw.to_list()
andes_p = Sb*np.array(ss.PV.p.v) 

print(pp_p[3])
print(andes_p[3])

x = np.arange(len(andes_p))
w = 0.35
plt.figure(figsize=(max(8, 0.35*len(x)), 4))
plt.grid()
plt.bar(x - w/2, pp_p, width=w, label='pandapower')
plt.bar(x + w/2, andes_p, width=w, label='ANDES')
plt.xticks(x, ss.PV.name.v, rotation=90, fontsize=8)
plt.xlabel("Generator Name")
plt.ylabel("Active Power (MW)")
plt.title('Plot gen active power')
plt.legend()
#plt.savefig('afb/P_gen.svg')
plt.show()

100.0
100.0


In [51]:
pp_q = net.res_gen.q_mvar.to_list() + net.res_sgen.q_mvar.to_list()
andes_q = Sb*np.array(ss.PV.q.v)

print(pp_q[3])
print(andes_q[3])

x = np.arange(len(andes_q))
w = 0.35
plt.figure(figsize=(max(8, 0.35*len(x)), 4))
plt.grid()
plt.bar(x - w/2, pp_q, width=w, label='pandapower')
plt.bar(x + w/2, andes_q, width=w, label='ANDES')
plt.xticks(x, ss.PV.name.v, rotation=90, fontsize=8)
plt.xlabel("Generator Name")
plt.ylabel("Reactive Power (Mvar)")
plt.title('Plot gen reactive power')
plt.legend()
#plt.savefig('afb/Q_gen.svg')
plt.show()

215.691677526891
215.691758946874


### Lines

In [52]:
pp_lpf = net.res_line.p_from_mw.to_list()
andes_lpf = Sb*np.array(df["P From"])

print(pp_lpf[3])
print(andes_lpf[3])

x = np.arange(len(pp_lpf))
w = 0.35
plt.figure(figsize=(max(8, 0.3*len(x)), 4.5))
plt.grid()
plt.bar(x - w/2, pp_lpf, width=w, label='pandapower')
plt.bar(x + w/2, andes_lpf[0:-4], width=w, label='ANDES')
plt.xlabel("Line Name")
plt.ylabel("P From (MW)")
plt.title('Plot P From')
plt.legend()
#plt.savefig('afb/P_From.svg')
plt.show()

91.49897314296028
13.871


In [53]:
pp_lqf = net.res_line.q_from_mvar.to_list()
andes_lqf = Sb*np.array(df["Q From"])

print(pp_lqf[3])
print(andes_lqf[3])

x = np.arange(len(pp_lqf))
w = 0.35
plt.figure(figsize=(max(8, 0.3*len(x)), 4.5))
plt.grid()
plt.bar(x - w/2, pp_lqf, width=w, label='pandapower')
plt.bar(x + w/2, andes_lqf[0:-4], width=w, label='ANDES')
plt.xlabel("Line Name")
plt.ylabel("Q From (Mvar)")
plt.title('Plot Q From')
plt.legend()
#plt.savefig('afb/Q_From.svg')
plt.show()

0.09315205187629161
-18.0858

In [54]:
pp_lpt = net.res_line.p_to_mw.to_list()
andes_lpt = Sb*np.array(df["P To"])

print(pp_lpt[3])
print(andes_lpt[3])

x = np.arange(len(pp_lpt))
w = 0.35
plt.figure(figsize=(max(8, 0.3*len(x)), 4.5))
plt.grid()
plt.bar(x - w/2, pp_lpt, width=w, label='pandapower')
plt.bar(x + w/2, andes_lpt[0:-4], width=w, label='ANDES')
plt.xlabel("Line Name")
plt.ylabel("P To (MW)")
plt.title('Plot P To')
plt.legend()
#plt.savefig('afb/P_To.svg')
plt.show()

-91.36931854411007
-13.8476


In [55]:
pp_lqt = net.res_line.q_to_mvar.to_list()
andes_lqt = Sb*np.array(df["Q To"])

print(pp_lqt[3])
print(andes_lqt[3])

x = np.arange(len(pp_lqt))
w = 0.35
plt.figure(figsize=(max(8, 0.3*len(x)), 4.5))
plt.grid()
plt.bar(x - w/2, pp_lqt, width=w, label='pandapower')
plt.bar(x + w/2, andes_lqt[0:-4], width=w, label='ANDES')
plt.xlabel("Line Name")
plt.ylabel("Q To (Mvar)")
plt.title('Plot Q To')
plt.legend()
#plt.savefig('afb/Q_To.svg')
plt.show()

-3.4294157109488914
16.7862
